In [65]:
# ==========================================================
# Clinical Event Prediction Model
# ==========================================================

import warnings
warnings.filterwarnings("ignore")

import time
import joblib
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.preprocessing import LabelEncoder

from sklearn.model_selection import train_test_split

from sklearn.metrics import (

    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix

)

from sklearn.ensemble import RandomForestClassifier

from lightgbm import LGBMClassifier

from xgboost import XGBClassifier

from catboost import CatBoostClassifier

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


In [66]:
# ==========================================================
# Load Dataset
# ==========================================================

DATASET_PATH = Path("../data/generated/master_dataset.csv")

df = pd.read_csv(

    DATASET_PATH,

    keep_default_na=False

)

print("="*60)

print("DATASET LOADED")

print("="*60)

print(df.shape)

df.head()

ParserError: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.

In [ ]:
print("="*60)

print("DATASET INFORMATION")

print("="*60)

print()

print("Rows :", df.shape[0])

print("Columns :", df.shape[1])

print()

print("Missing :", df.isnull().sum().sum())

print("Duplicate :", df.duplicated().sum())

DATASET INFORMATION

Rows : 100000
Columns : 113

Missing : 0
Duplicate : 0


In [ ]:
print("="*60)

print("DATASET INFORMATION")

print("="*60)

print()

print("Rows :", df.shape[0])

print("Columns :", df.shape[1])

print()

print("Missing :", df.isnull().sum().sum())

print("Duplicate :", df.duplicated().sum())

DATASET INFORMATION

Rows : 100000
Columns : 113

Missing : 0
Duplicate : 0


In [ ]:
# ==========================================================
# Target Variable
# ==========================================================

TARGET = "Possible_Event"

print("Target :", TARGET)

print()

print(df[TARGET].value_counts())

Target : Possible_Event

Possible_Event
Stable                  71881
Cardiac Event Risk       9600
Metabolic Risk           8863
Respiratory Distress     6442
Hypertensive Crisis      2272
Possible Stroke           942
Name: count, dtype: int64


In [ ]:
# ==========================================================
# Remove Leakage Columns
# ==========================================================

LEAKAGE_COLUMNS = [

    "Risk_Level",

    "Alert_Level",

    "Recommendation",

    "Confidence",

    "Clinical_Score"

]

X = df.drop(

    columns=LEAKAGE_COLUMNS + [TARGET],

    errors="ignore"

)

y = df[TARGET]

print("Features :", X.shape)

print("Target :", y.shape)

Features : (100000, 107)
Target : (100000,)


In [ ]:
# ==========================================================
# Remove Engineered Risk Features
# ==========================================================

ENGINEERED_COLUMNS = [

    "Cardiovascular_Risk",
    "Respiratory_Risk",
    "Neurological_Risk",
    "Metabolic_Risk",
    "Lifestyle_Risk",
    "Trend_Risk",

    "Cardiovascular_Score",
    "Respiratory_Score",
    "Neurological_Score",
    "Metabolic_Score",
    "Mobility_Score",
    "Sleep_Score",

    "Health_Index"

]

X = X.drop(

    columns=ENGINEERED_COLUMNS,

    errors="ignore"

)

print("Remaining Features :", X.shape)

Remaining Features : (100000, 94)


In [ ]:
# ==========================================================
# Encode Features
# ==========================================================

feature_encoders = {}

categorical_columns = X.select_dtypes(include="object").columns

for column in categorical_columns:

    encoder = LabelEncoder()

    X[column] = encoder.fit_transform(
        X[column].astype(str)
    )

    feature_encoders[column] = encoder

target_encoder = LabelEncoder()

y = target_encoder.fit_transform(y)

print("Encoding Complete")

print("Features :", X.shape)

Encoding Complete
Features : (100000, 94)


In [ ]:
# ==========================================================
# Train Test Split
# ==========================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.20,

    random_state=42,

    stratify=y

)

print("Train :", X_train.shape)

print("Test :", X_test.shape)

Train : (80000, 94)
Test : (20000, 94)


In [ ]:
# ==========================================================
# Model Library
# ==========================================================

MODELS = {

    "Random Forest": RandomForestClassifier(

        n_estimators=300,
        max_depth=20,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1

    ),

    "XGBoost": XGBClassifier(

        n_estimators=300,
        learning_rate=0.05,
        max_depth=8,
        random_state=42,
        eval_metric="mlogloss",
        tree_method="hist",
        n_jobs=-1

    ),

    "LightGBM": LGBMClassifier(

        n_estimators=300,
        learning_rate=0.05,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1

    ),

    "CatBoost": CatBoostClassifier(

        iterations=300,
        learning_rate=0.05,
        depth=8,
        random_seed=42,
        verbose=False

    )

}

print("Models Ready")

Models Ready


In [ ]:
# ==========================================================
# Train Models
# ==========================================================

results = {}

trained_models = {}

for model_name, model in MODELS.items():

    print("\n" + "="*70)

    print(f"Training {model_name}")

    print("="*70)

    start = time.time()

    model.fit(X_train, y_train)

    training_time = time.time() - start

    predictions = model.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)

    precision = precision_score(
        y_test,
        predictions,
        average="weighted",
        zero_division=0
    )

    recall = recall_score(
        y_test,
        predictions,
        average="weighted",
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        predictions,
        average="weighted",
        zero_division=0
    )

    results[model_name] = {

        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Training_Time": training_time

    }

    trained_models[model_name] = model

    print(f"Accuracy : {accuracy:.4f}")

    print(f"F1 Score : {f1:.4f}")


Training Random Forest
Accuracy : 0.9952
F1 Score : 0.9952

Training XGBoost
Accuracy : 0.9998
F1 Score : 0.9998

Training LightGBM
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.021832 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2927
[LightGBM] [Info] Number of data points in the train set: 80000, number of used features: 94
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Info] Start training from score -1.791760
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits w

In [ ]:
# ==========================================================
# Model Comparison
# ==========================================================

comparison = pd.DataFrame(results).T

comparison = comparison.sort_values(

    by="F1",

    ascending=False

)

comparison

,Accuracy,Precision,Recall,F1,Training_Time
CatBoost,0.99990,0.999900,0.99990,0.999900,67.992764
LightGBM,0.99985,0.999850,0.99985,0.999850,12.483427
XGBoost,0.99980,0.999800,0.99980,0.999800,14.773877
Random Forest,0.99525,0.995232,0.99525,0.995227,22.653067


In [ ]:
# ==========================================================
# Best Model
# ==========================================================

best_model_name = comparison.index[0]

best_model = trained_models[best_model_name]

print("="*60)

print("BEST MODEL")

print("="*60)

print(best_model_name)

print()

print(comparison.loc[best_model_name])

BEST MODEL
CatBoost

Accuracy          0.999900
Precision         0.999900
Recall            0.999900
F1                0.999900
Training_Time    67.992764
Name: CatBoost, dtype: float64


In [ ]:
# ==========================================================
# Save Model
# ==========================================================

MODEL_DIR = Path("../models")

MODEL_DIR.mkdir(

    parents=True,

    exist_ok=True

)

joblib.dump(

    best_model,

    MODEL_DIR / "clinical_event_model.pkl"

)

joblib.dump(

    target_encoder,

    MODEL_DIR / "clinical_event_label_encoder.pkl"

)

joblib.dump(

    feature_encoders,

    MODEL_DIR / "clinical_event_feature_encoders.pkl"

)

print("Clinical Event Model Saved Successfully")

Clinical Event Model Saved Successfully


In [ ]:
pred = best_model.predict(X_test)

print(
    classification_report(
        y_test,
        pred,
        target_names=target_encoder.classes_
    )
)

                      precision    recall  f1-score   support

  Cardiac Event Risk       1.00      1.00      1.00      1920
 Hypertensive Crisis       1.00      1.00      1.00       454
      Metabolic Risk       1.00      1.00      1.00      1773
     Possible Stroke       1.00      1.00      1.00       189
Respiratory Distress       1.00      1.00      1.00      1288
              Stable       1.00      1.00      1.00     14376

            accuracy                           1.00     20000
           macro avg       1.00      1.00      1.00     20000
        weighted avg       1.00      1.00      1.00     20000



In [ ]:
cm = confusion_matrix(y_test, pred)

cm_df = pd.DataFrame(
    cm,
    index=target_encoder.classes_,
    columns=target_encoder.classes_
)

cm_df

,Cardiac Event Risk,Hypertensive Crisis,Metabolic Risk,Possible Stroke,Respiratory Distress,Stable
Cardiac Event Risk,1920,0,0,0,0,0
Hypertensive Crisis,0,454,0,0,0,0
Metabolic Risk,0,0,1771,0,0,2
Possible Stroke,0,0,0,189,0,0
Respiratory Distress,0,0,0,0,1288,0
Stable,0,0,0,0,0,14376


In [ ]:
print("=" * 70)
print("Clinical Event Training Dataset")
print("=" * 70)

print("Train Shape :", X_train.shape)
print("Test Shape  :", X_test.shape)

print()

print("Total Features :", X_train.shape[1])

Clinical Event Training Dataset
Train Shape : (80000, 94)
Test Shape  : (20000, 94)

Total Features : 94


In [ ]:
feature_df = pd.DataFrame({
    "Feature": X_train.columns
})

feature_df.to_csv(
    "clinical_event_features.csv",
    index=False
)

feature_df

,Feature
0,Patient_ID
1,Age
2,Age_Group
3,Gender
4,Height_cm
...,...
89,Fever_Flag
90,Low_Oxygen_Flag
91,Tachycardia_Flag
92,Hypertension_Flag


In [ ]:
feature_df = pd.DataFrame({
    "Feature": X_train.columns
})

feature_df.to_csv(
    "clinical_event_features.csv",
    index=False
)

print("Saved Successfully")

Saved Successfully


In [ ]:
# ==========================================================
# Remove Non-Predictive Features
# ==========================================================

DROP_COLUMNS = [

    "Patient_ID",

    "Timestamp"

]

X_train = X_train.drop(
    columns=DROP_COLUMNS,
    errors="ignore"
)

X_test = X_test.drop(
    columns=DROP_COLUMNS,
    errors="ignore"
)

print("=" * 60)
print("Columns Removed")
print("=" * 60)

print()

print("Train :", X_train.shape)
print("Test  :", X_test.shape)

Columns Removed

Train : (80000, 92)
Test  : (20000, 92)


In [ ]:
# ==========================================================
# Initialize Models
# ==========================================================

MODELS = {

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=20,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost": XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=8,
        random_state=42,
        eval_metric="mlogloss",
        tree_method="hist"
    ),

    "LightGBM": LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        class_weight="balanced",
        random_state=42
    ),

    "CatBoost": CatBoostClassifier(
        iterations=300,
        learning_rate=0.05,
        depth=8,
        random_seed=42,
        verbose=False
    )

}

print("Models Initialized Successfully")

Models Initialized Successfully


In [ ]:
# ==========================================================
# Train Clinical Event Models
# ==========================================================

results = {}

trained_models = {}

for model_name, model in MODELS.items():

    print("=" * 70)
    print(f"TRAINING {model_name.upper()}")
    print("=" * 70)

    start = time.time()

    model.fit(X_train, y_train)

    training_time = time.time() - start

    predictions = model.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)

    precision = precision_score(
        y_test,
        predictions,
        average="weighted"
    )

    recall = recall_score(
        y_test,
        predictions,
        average="weighted"
    )

    f1 = f1_score(
        y_test,
        predictions,
        average="weighted"
    )

    results[model_name] = {

        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Training_Time": training_time

    }

    trained_models[model_name] = model

    print()
    print(f"Accuracy : {accuracy:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print()

TRAINING RANDOM FOREST

Accuracy : 0.9958
F1 Score : 0.9957

TRAINING XGBOOST

Accuracy : 0.9998
F1 Score : 0.9998

TRAINING LIGHTGBM
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.013893 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2652
[LightGBM] [Info] Number of data points in the train set: 80000, number of used features: 92
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Info] Start training from score -1.791760
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits 

In [ ]:
comparison = pd.DataFrame(results).T

comparison = comparison.sort_values(
    by="Accuracy",
    ascending=False
)

comparison

,Accuracy,Precision,Recall,F1,Training_Time
LightGBM,0.99990,0.999900,0.99990,0.999900,9.113970
XGBoost,0.99980,0.999800,0.99980,0.999800,12.295960
CatBoost,0.99980,0.999800,0.99980,0.999800,62.309153
Random Forest,0.99575,0.995737,0.99575,0.995735,18.874076


In [67]:
# ==========================================================
# Clinical Event Classification Report
# ==========================================================

from sklearn.metrics import classification_report

predictions = best_model.predict(X_test)

print("=" * 70)
print(f"{best_model_name.upper()} CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        y_test,
        predictions
    )
)

CatBoostError: catboost/libs/data/model_dataset_compatibility.cpp:81: At position 1 should be feature with name Age (found Age_Group).

In [68]:
# ==========================================================
# Clinical Event Classification Report
# ==========================================================

from sklearn.metrics import classification_report

predictions = best_model.predict(X_test)

print("=" * 70)
print(f"{best_model_name.upper()} CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        y_test,
        predictions
    )
)

CatBoostError: catboost/libs/data/model_dataset_compatibility.cpp:81: At position 1 should be feature with name Age (found Age_Group).

In [69]:
print(best_model_name)
print(type(best_model))

CatBoost
<class 'catboost.core.CatBoostClassifier'>


In [70]:
# ==========================================================
# Select Best Model
# ==========================================================

comparison = comparison.sort_values(
    by="Accuracy",
    ascending=False
)

best_model_name = comparison.index[0]

best_model = trained_models[best_model_name]

print(best_model_name)
print(type(best_model))

LightGBM
<class 'lightgbm.sklearn.LGBMClassifier'>


In [71]:
# ==========================================================
# Clinical Event Classification Report
# ==========================================================

from sklearn.metrics import classification_report

predictions = best_model.predict(X_test)

print("=" * 70)
print(f"{best_model_name.upper()} CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        y_test,
        predictions
    )
)

LIGHTGBM CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1920
           1       1.00      1.00      1.00       454
           2       1.00      1.00      1.00      1773
           3       1.00      1.00      1.00       189
           4       1.00      1.00      1.00      1288
           5       1.00      1.00      1.00     14376

    accuracy                           1.00     20000
   macro avg       1.00      1.00      1.00     20000
weighted avg       1.00      1.00      1.00     20000



In [72]:
# ==========================================================
# Clinical Event Confusion Matrix
# ==========================================================

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, predictions)

cm_df = pd.DataFrame(cm)

cm_df

,0,1,2,3,4,5
0,1920,0,0,0,0,0
1,0,454,0,0,0,0
2,0,0,1771,0,0,2
3,0,0,0,189,0,0
4,0,0,0,0,1288,0
5,0,0,0,0,0,14376


In [73]:
# ==========================================================
# Clinical Event Feature Importance
# ==========================================================

importance = pd.DataFrame({

    "Feature": X_train.columns,

    "Importance": best_model.feature_importances_

})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

importance.head(25)

,Feature,Importance
68,SpO2,5028
80,Respiratory_Deviation,4500
75,Heart_Rate_Deviation,4168
66,Systolic_BP,3188
76,Systolic_BP_Deviation,3187
8,Diabetes,2975
47,Uses_Salbutamol,1878
71,Sleep_Hours,1325
12,Heart_Disease,1269
91,Vital_Abnormality_Count,1131


In [74]:
# ==========================================================
# Save Final Clinical Event Model
# ==========================================================

from pathlib import Path
import joblib

MODEL_DIR = Path("../models")

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

joblib.dump(
    best_model,
    MODEL_DIR / "clinical_event_model_v2.pkl"
)

print("=" * 60)
print("Clinical Event Model Saved Successfully")
print("=" * 60)

print("Model :", best_model_name)

Clinical Event Model Saved Successfully
Model : LightGBM


In [75]:
# ==========================================================
# Save Clinical Event Feature Columns
# ==========================================================

joblib.dump(
    list(X_train.columns),
    MODEL_DIR / "clinical_event_feature_columns_v2.pkl"
)

print("Clinical Event Feature Columns Saved")

Clinical Event Feature Columns Saved


In [76]:
import os

print("=" * 60)
print("Models Folder")
print("=" * 60)

for file in sorted(os.listdir(MODEL_DIR)):
    print(file)

Models Folder
alert_feature_encoders.pkl
alert_features.pkl
alert_label_encoder.pkl
alert_scaler.pkl
best_health_model.pkl
clinical_event_feature_columns_v2.pkl
clinical_event_feature_encoders.pkl
clinical_event_label_encoder.pkl
clinical_event_model.pkl
clinical_event_model_v2.pkl
disease_models
encoders
health_feature_columns_v2.pkl
health_feature_encoders.pkl
health_features.pkl
health_label_encoder.pkl
health_risk_model_v2.pkl
health_scaler.pkl
